# Функции в Python

## Цели лекции

После этой лекции вы сможете:
- определять функции с параметрами, значениями по умолчанию и `*args`/`**kwargs`;
- ограничивать способ передачи аргументов через `/` и `*`;
- возвращать результат через `return`, в том числе несколько значений сразу
  (и понимать, что без `return` функция возвращает `None`);
- объяснять, почему функция может незаметно испортить переданный ей список;
- писать анонимные функции `lambda` и объяснять полиморфизм в Python;
- передавать функции аргументами и возвращать их из функций;
- писать рекурсивные функции и понимать, где вместо них лучше цикл;
- снабжать функции аннотациями типов и докстрингами;
- отличать чистую функцию от нечистой и отделять вычисление от побочного эффекта;
- писать простые тесты функций на `pytest`;
- **избегать классической ошибки** с изменяемым значением по умолчанию.


### Основы функций в Python
- Функция – это именованный блок кода, который можно многократно вызывать.  
- Использование функций предотвращает дублирование кода (альтернатива «копировать-вставить»).  
- Функции помогают разбивать сложные системы на управляемые части.  
- Этот принцип часто называют *divide and conquer* («разделяй и властвуй»).  
- Функции могут выступать как основа для реализации **методов объектов (object methods)** в классах.  

### def и return в функциях
- `def` создаёт объект функции и присваивает его имени.  
- Заголовок (`def name(args):`) + тело (внутренний блок).  
- Вызов: `name(values)` – запускает тело функции.  
- Параметры в заголовке при вызове связываются с переданными значениями – аргументами.  
- `return` завершает работу функции и возвращает результат.  
- Если `return` отсутствует → функция возвращает `None`.  

In [4]:
# Функция с return
def square(x):
    return x ** 2

result = square(5)
print("Квадрат числа:", result)

Квадрат числа: 25


In [5]:
# Функция без return (возвращает None)
def printer(msg):
    print("Message:", msg)

val = printer("Hi")
print("Возвращаемое значение:", val)

Message: Hi
Возвращаемое значение: None


### Ранний возврат уменьшает вложенность

Особые случаи стоит обрабатывать в начале функции и сразу возвращать результат:
тогда основное действие остаётся ниже, без цепочки вложенных `else`.

После выполненного `return` остальные инструкции тела функции не выполняются.


In [1]:
def entry_message(is_open, has_ticket):
    if not is_open:
        return "Закрыто"
    if not has_ticket:
        return "Нужен билет"
    return "Проходите"


print(entry_message(False, True))
print(entry_message(True, False))
print(entry_message(True, True))


Закрыто
Нужен билет
Проходите


### `return` внутри цикла завершает всю функцию

При поиске первого совпадения вернуть индекс можно сразу.
Но сделать вывод «не найдено» можно только **после всего обхода**.

В ошибочной версии `return None` стоит внутри цикла: поиск заканчивается
уже на первом неподходящем элементе.


In [2]:
def find_position_bad(items, target):
    for index, item in enumerate(items):
        if item == target:
            return index
        return None

def find_position(items, target):
    for index, item in enumerate(items):
        if item == target:
            return index
    return None

print("Ошибка:", find_position_bad(["чай", "кофе"], "кофе"))
print("Исправлено:", find_position(["чай", "кофе"], "кофе"))


Ошибка: None
Исправлено: 1


### Передача аргументов: объект, а не копия

При вызове `spoil(data)` Python первым делом выполняет присваивание
`items = data`, где `items` – параметр функции. Получаются совместные ссылки,
как после `L2 = L1` в лекции 3: копии нет, оба имени указывают на один список.
Совпадают ли имена снаружи и внутри функции, значения не имеет.

Что увидит вызывающий код, решает операция внутри функции:

- **присваивание имени** (`items = [...]`) переставляет на новый объект только
  локальное имя `items`, а `data` по-прежнему указывает на старый список;
- **изменение объекта** (`items[0] = 99`, `items.append(4)`) меняет сам список,
  и это видно через `data`.

Числа, строки и кортежи на месте не меняются, поэтому через такой аргумент
функция вызывающий код не затронет. Изменяемый аргумент она может испортить
незаметно.

In [6]:
def rename(items):
    items = ["другой список"]    # присваивание: имя items указывает на новый список
    print("внутри rename:", items)

def spoil(items):
    items[0] = 99                # изменение: меняется сам список, общий с data
    items.append(4)

data = [1, 2, 3]
rename(data)
print("после rename:", data)     # data указывает на прежний список
spoil(data)
print("после spoil: ", data)     # тот же список, но изменённый

внутри rename: ['другой список']
после rename: [1, 2, 3]
после spoil:  [99, 2, 3, 4]


In [7]:
# Защита – передать копию: функция изменит копию, исходный список останется прежним
data = [1, 2, 3]
spoil(data[:])
print(data)

[1, 2, 3]


In [8]:
# Два способа написать одно и то же преобразование
def add_tax_inplace(prices):
    for i, price in enumerate(prices):
        prices[i] = round(price * 1.2, 2)

def with_tax(prices):
    return [round(price * 1.2, 2) for price in prices]

original = [100, 200]
print(with_tax(original), original)   # исходные данные целы

add_tax_inplace(original)
print(original)                       # исходных данных больше нет

[120.0, 240.0] [100, 200]
[120.0, 240.0]


### Параметры функции: позиционные и именованные

**Сигнатура** функции – это её параметры и правила вызова: что обязательно,
что имеет значение по умолчанию, что можно передать по имени.

Аргументы можно передавать по **позиции** и по **имени**, а параметрам задавать
**значения по умолчанию**.

- Среди позиционных параметров те, что со значением по умолчанию, идут после
  обязательных. Для параметров, передаваемых только по имени (о них дальше,
  в разделе про `/` и `*`), порядок свободный.
- Именованные аргументы можно передавать в любом порядке.
- Со значениями по умолчанию связана известная ловушка, к которой вернёмся
  в «Частых ошибках».

In [9]:
def order(item, count=1, note="без пожеланий"):
    return f"{count} x {item} ({note})"


print(order("кофе"))                      # только обязательный
print(order("кофе", 3))                   # по позиции
print(order("кофе", note="без сахара"))   # пропускаем средний
print(order(count=2, item="чай"))         # порядок не важен


1 x кофе (без пожеланий)
3 x кофе (без пожеланий)
1 x кофе (без сахара)
2 x чай (без пожеланий)


### Ошибки связывания аргументов

Перед выполнением тела Python связывает переданные аргументы с параметрами.
Вызов не состоится, если обязательного аргумента не хватает или передано
неизвестное имя.

В обоих случаях возникает `TypeError`. Сообщение указывает на проблемный параметр.


In [3]:
def label(item, count=1):
    print("Тело label выполняется")
    return f"{item}: {count} шт."

try:
    label()
except TypeError as error:
    print(error)

try:
    label("чай", copies=2)
except TypeError as error:
    print(error)


label() missing 1 required positional argument: 'item'
label() got an unexpected keyword argument 'copies'


### Один параметр нельзя передать дважды

Вызовы `label("чай", 2)` и `label("чай", count=2)` допустимы.
Но `label("чай", 2, count=3)` даёт два значения параметру `count`.

Именованный аргумент не заменяет уже переданное позиционное значение.
Python выбрасывает `TypeError` до выполнения тела функции.


In [4]:
print(label("чай", count=2))

try:
    label("чай", 2, count=3)
except TypeError as error:
    print(error)


Тело label выполняется
чай: 2 шт.
label() got multiple values for argument 'count'


### Переменное число аргументов: `*args` и `**kwargs`

- `*args` собирает лишние **позиционные** аргументы в кортеж.
- `**kwargs` собирает лишние **именованные** в словарь.
- Имена условные: значение имеют звёздочки, а не слова `args` и `kwargs`.

Те же звёздочки работают и **при вызове**, только в обратную сторону:
раскладывают коллекцию по аргументам. Это та же распаковка, что разбиралась
в лекции про инструкции.

In [10]:
def report(title, *values, **options):
    print(f"{title}: значений {len(values)}, настроек {len(options)}")
    print("  values =", values)      # кортеж
    print("  options =", options)    # словарь


report("Замер", 1, 2, 3, unit="мс", precision=2)


Замер: значений 3, настроек 2
  values = (1, 2, 3)
  options = {'unit': 'мс', 'precision': 2}


In [11]:
# Звёздочки при вызове: раскладываем готовые коллекции по аргументам
nums = [10, 20, 30]
cfg = {"unit": "кг", "precision": 1}

report("Из коллекций", *nums, **cfg)


Из коллекций: значений 3, настроек 2
  values = (10, 20, 30)
  options = {'unit': 'кг', 'precision': 1}


### Явный аргумент может конфликтовать с `**словарём`

После распаковки ключи словаря становятся именованными аргументами того же вызова.
Если `count` передан и явно, и через словарь, Python не выбирает ни одно из двух
значений: вызов падает с `TypeError`, как при двойной передаче выше.

Чтобы переопределить настройку, нужное значение записывают в копию словаря
и передают в функцию только её.


In [5]:
options = {"count": 2}

# count приходит и явно, и из словаря. Связывание падает до выполнения тела,
# поэтому строки "Тело label выполняется" в этом случае нет
try:
    label("чай", count=3, **options)
except TypeError as error:
    print(error)

# Правильный путь: подготовить словарь заранее и передать его один раз
updated_options = options.copy()
updated_options["count"] = 3
print(label("чай", **updated_options))   # сначала печатает тело, затем результат

print("Исходные настройки:", options)    # copy() уберёг исходный словарь

label() got multiple values for keyword argument 'count'
Тело label выполняется
чай: 3 шт.
Исходные настройки: {'count': 2}


### Как запретить способ передачи: `/` и `*`

В сигнатуре можно ограничить, как именно передаются аргументы:

- всё **до** `/` – только по позиции (positional-only);
- всё **после** `*` – только по имени (keyword-only).

Зачем: имя позиционного параметра перестаёт быть частью публичного интерфейса –
его можно переименовать, не ломая чужой код. А keyword-only заставляет писать
`convert(36.6, to="fahrenheit")` вместо загадочного `convert(36.6, "fahrenheit")`.

Такие сигнатуры часто встречаются в стандартной библиотеке: например,
`sorted(iterable, /, *, key=None, reverse=False)`.

In [12]:
def convert(value, /, *, to="celsius"):
    return f"{value} -> {to}"


print(convert(36.6, to="fahrenheit"))

try:
    convert(value=36.6)              # value – только по позиции
except TypeError as e:
    print("TypeError:", e)

try:
    convert(36.6, "fahrenheit")      # to – только по имени
except TypeError as e:
    print("TypeError:", e)


36.6 -> fahrenheit
TypeError: convert() got some positional-only arguments passed as keyword arguments: 'value'
TypeError: convert() takes 1 positional argument but 2 were given


### Полный порядок параметров

Все виды параметров в одной сигнатуре, в том порядке, в котором они обязаны идти:

```python
def show(a, /, b, c=0, *args, d, e=1, **kwargs): ...
```

| Группа | В примере | Как передаётся |
| --- | --- | --- |
| только позиционные | `a` | только по позиции |
| обычные | `b`, `c=0` | по позиции или по имени |
| лишние позиционные | `*args` | собираются в кортеж |
| только именованные | `d`, `e=1` | только по имени |
| лишние именованные | `**kwargs` | собираются в словарь |

Любую группу можно пропустить, переставлять нельзя. Если `*args` не нужен,
только именованные параметры отделяют одиночной `*`, как в `convert`.

Вызов Python разбирает в два прохода:

1. Позиционные аргументы заполняют параметры слева направо: `a`, `b`, `c`.
   Остаток уходит в `args`.
2. Именованные аргументы ищут параметр со своим именем. Имена, которых
   в сигнатуре нет, уходят в `kwargs`.

Параметр, которому ничего не досталось, получает значение по умолчанию,
а если его нет – `TypeError`. Лишние аргументы, когда `*args` или `**kwargs`
в сигнатуре нет, – тоже `TypeError`.

In [15]:
# Куда попадает каждый аргумент вызова
def show(a, /, b, c=0, *args, d, e=1, **kwargs):
    print(f"a={a} b={b} c={c} args={args} d={d} e={e} kwargs={kwargs}")


show(1, 2, d=4)                   # минимум: a и b по позиции, d по имени
show(1, 2, 3, 4, 5, d=6, x=7)     # 1, 2, 3 заняли a, b, c; 4 и 5 – в args; x – в kwargs
show(1, c=3, b=2, e=5, d=4)       # обычные b и c можно и по имени, в любом порядке

try:
    show(1, 2, 3, 4)              # 4 уходит в args, а не в d: d – только по имени
except TypeError as error:
    print("TypeError:", error)

a=1 b=2 c=0 args=() d=4 e=1 kwargs={}
a=1 b=2 c=3 args=(4, 5) d=6 e=1 kwargs={'x': 7}
a=1 b=2 c=3 args=() d=4 e=5 kwargs={}
TypeError: show() missing 1 required keyword-only argument: 'd'


In [16]:
# Со значением по умолчанию ошибки нет: лишний аргумент молча уходит в *args
def join_words(*words, sep=" "):
    return sep.join(words)

print(join_words("чай", "кофе", sep="-"))
print(join_words("чай", "кофе", "-"))       # "-" попал в words, а не в sep

# Так же устроен print(*objects, sep=" ", end="\n"): поэтому sep и end
# передают только по имени

чай-кофе
чай кофе -


### Возврат нескольких значений

`return a, b, c` возвращает **кортеж** – отдельного механизма «несколько
возвращаемых значений» в Python нет. На принимающей стороне работает обычная
распаковка из лекции про инструкции.

In [13]:
def min_max_avg(numbers):
    return min(numbers), max(numbers), sum(numbers) / len(numbers)


low, high, avg = min_max_avg([4, 8, 15, 16, 23, 42])
print(f"min={low} max={high} avg={avg:.2f}")

result = min_max_avg([1, 2, 3])
print("на самом деле кортеж:", result, type(result).__name__)


min=4 max=42 avg=18.00
на самом деле кортеж: (1, 3, 2.0) tuple


### Индекс `0` – результат, `None` – отсутствие совпадения

`find_position` из примера про `return` внутри цикла возвращает индекс первого
совпадения, а если совпадения нет (в том числе в пустом списке) – `None`.

Подвох – в проверке результата. Индекс `0` – законная находка: элемент стоит
первым. Но `bool(0)` равно `False`, поэтому `if position:` примет такую находку
за «не найдено». Отсутствие результата проверяют явно: **`if position is None:`**.


In [6]:
# Неверная проверка: находка с индексом 0 теряется
position = find_position(["чай", "кофе"], "чай")
if position:                     # bool(0) – False
    print("чай – индекс", position)
else:
    print("чай – не найдено (ошибка: чай стоит первым)")

# Верная проверка: None сравнивается явно
for wanted in ["чай", "вода"]:
    position = find_position(["чай", "кофе"], wanted)
    if position is None:
        print(wanted, "– не найдено")
    else:
        print(wanted, "– индекс", position)

print("Пустой вход:", find_position([], "чай"))


чай – не найдено (ошибка: чай стоит первым)
чай – индекс 0
вода – не найдено
Пустой вход: None


### Неверный запрос – исключение, нет совпадения – `None`

Функция может не найти ответ по двум разным причинам, и смешивать их нельзя:

- **запрос корректен, но совпадения нет** (ищем `"вода"`, а её нет в списке) –
  обычный исход, функция возвращает `None`;
- **запрос некорректен** (пустая строка) – искать нечего, ошибся вызывающий код.
  Функция выбрасывает `ValueError`: если бы она вернула `None`, ошибка выглядела
  бы как «товар не найден» и прошла бы незамеченной.

Какой случай как обрабатывается, записывают в докстринге и проверяют тестами.


In [7]:
def find_product(names, query):
    """Вернуть индекс товара или None; пустой запрос вызывает ValueError."""
    if not query.strip():
        raise ValueError("Запрос не должен быть пустым")
    return find_position(names, query)


print("Товара нет:", find_product(["чай"], "вода"))
try:
    find_product(["чай"], "  ")
except ValueError as error:
    print("Неверный запрос:", error)


Товара нет: None
Неверный запрос: Запрос не должен быть пустым


### Выполнение `def` и выражение `lambda`
- `def` – **исполняемая инструкция**, а не объявление: пока строка не выполнилась, функции не существует.  
- Поэтому функции можно определять не только на верхнем уровне, но и внутри `if`, циклов или других функций.  
- Имя функции – это всего лишь ссылка на объект. Можно присвоить функцию другой переменной и вызывать её через новое имя.  
- К функциям можно добавлять произвольные атрибуты.  
- `lambda` создаёт **анонимную функцию**, без имени: `lambda args: expression`.  

In [14]:
# def выполняется при достижении инструкции
test = True
if test:
    def func():
        return "Версия 1"
else:
    def func():
        return "Версия 2"

print(func())     # "Версия 1": выполнилась только ветка if, второй def не создан

# Имя функции – просто ссылка
other = func
print(other())    # тот же объект под другим именем, снова "Версия 1"

Версия 1
Версия 1


In [15]:
# Атрибуты у функции
func.attr = "данные"
print(func.attr)

данные


In [20]:
# lambda – функция-выражение: то же, что def square выше, но без имени
print((lambda x: x ** 2)(5))

# Объект того же типа, что создаёт def, только имя у него служебное
print(type(lambda x: x ** 2) is type(square))
print(square.__name__, (lambda x: x ** 2).__name__)

25
True
square <lambda>


### Границы `lambda`

`lambda` – это **выражение**, а не инструкция. В теле допустимо только то, что
само является выражением: арифметика, вызовы, тернарный `if`, включения.
Присваивание, `try` и `return` внутрь не помещаются – для них нужен `def`.

Про стиль: PEP 8 не рекомендует присваивать `lambda` имени (`f = lambda x: ...`),
и линтеры выдают на такую строку предупреждение (в `ruff` это правило E731).
Если функции нужно имя, у Python для этого есть `def`: он примет докстринг,
а в трассировке ошибки покажет настоящее имя вместо `<lambda>`.

In [23]:
# В теле lambda допустимо только выражение
print(list(map(lambda n: "чёт" if n % 2 == 0 else "нечет", [5, 8, 3])))

['нечет', 'чёт', 'нечет']


### Проверка синтаксиса без запуска: `compile`

Инструкцию в теле `lambda` нельзя показать, просто записав её в ячейке:
`SyntaxError` возникает при разборе ячейки, до `try`, и не выполнится ни одна
строка. Поэтому код передаётся строкой во встроенную функцию `compile`. Она
переводит текст в исполняемый код, но не запускает его, а синтаксическая ошибка
обнаруживается уже на этом шаге – её можно поймать.

`compile(source, filename, mode)`:

- `source` – текст кода;
- `filename` – имя файла для сообщений об ошибках; файла нет, поэтому
  подставляем условное `"<lambda>"`;
- `mode` – что ожидается в тексте: `"eval"` – одно выражение, как тело `lambda`,
  `"exec"` – последовательность инструкций.

`{source!r}` в f-строке – то же, что `repr(source)` из лекции 3: строка
выводится в кавычках, как в коде.

In [24]:
for source in ["lambda n: if n > 0: return n", "lambda n: total = n + 1"]:
    try:
        compile(source, "<lambda>", "eval")
    except SyntaxError as e:
        print(f"{source!r} -> SyntaxError: {e.msg}")

'lambda n: if n > 0: return n' -> SyntaxError: invalid syntax
'lambda n: total = n + 1' -> SyntaxError: invalid syntax


### Полиморфизм в Python
- Полиморфизм – это зависимость результата операции от типов объектов, с которыми она выполняется.  
- В Python так устроены почти все операции, а не только `*`:
  - `+` складывает числа, но склеивает строки и списки;
  - `x[k]` у списка выбирает элемент по номеру, у словаря – значение по ключу;
  - `len(x)` считает символы строки, элементы списка, ключи словаря;
  - `print(x)` выводит объект любого типа: каждый тип сам задаёт свой текстовый вид.  
- Оператор или функция работают, если объекты поддерживают ожидаемый интерфейс (протокол).   
- Если объекты не поддерживают ожидаемую операцию, возникает исключение `TypeError`.  
- Обратная сторона той же гибкости – пример `area("10", 5)` из лекции про типы
  данных: функция без проверок повторила строку вместо умножения.  

In [39]:
def times(x, y):
    return x * y

# Полиморфизм в действии
print(times(3, 4))        # числа перемножаются
print(times(3.14, 4))     # с float работает так же
print(times("Py", 3))     # строка повторяется

# Граница: умножения строки на строку не существует
try:
    times("Py", "3")
except TypeError as error:
    print("TypeError:", error)

12
12.56
PyPyPy
TypeError: can't multiply sequence by non-int of type 'str'


## Функции высшего порядка

Функция – обычный объект: её уже присваивали другому имени. Значит, её можно
передать в другую функцию и вернуть из функции. **Функция высшего порядка** –
та, которая принимает функцию аргументом или возвращает функцию.

Мы этим уже пользовались: `sorted(key=...)`, `map`, `filter` – всё это функции
высшего порядка. Здесь пишем свои.

Возврат функции из функции – основа декораторов, к которым перейдём в следующей
лекции.

In [20]:
def apply_to_all(func, items):
    return [func(item) for item in items]


def shout(word):
    return word.upper() + "!"


print(apply_to_all(shout, ["раз", "два"]))
print(apply_to_all(len, ["раз", "два"]))            # встроенная функция тоже подойдёт
print(apply_to_all(lambda w: w[::-1], ["раз", "два"]))

['РАЗ!', 'ДВА!']
[3, 3]
['зар', 'авд']


In [21]:
# Правило поведения передаётся функцией – это избавляет от дублирования
def count_matching(items, rule):
    chosen = [x for x in items if rule(x)]
    return f"подошло {len(chosen)} из {len(items)}: {chosen}"


nums = [3, 8, 12, 5, 20]
print(count_matching(nums, lambda n: n > 10))
print(count_matching(nums, lambda n: n % 2 == 0))

подошло 2 из 5: [12, 20]
подошло 3 из 5: [8, 12, 20]


In [22]:
def make_multiplier(factor):
    def multiply(value):         # внутренняя функция помнит factor
        return value * factor
    return multiply              # возвращаем саму функцию, без вызова


double = make_multiplier(2)      # factor = 2, в double – функция multiply
triple = make_multiplier(3)      # factor = 3, ещё одна функция multiply

print(double(10), triple(10))    # value = 10: 10 * 2 и 10 * 3
print("имя объекта:", double.__name__)

20 30
имя объекта: multiply


### Рекурсия

Функция может вызывать саму себя – так был устроен `flatten` из лекции 4б.
Обязательное условие – **базовый случай**, при котором вызовы прекращаются;
без него рекурсия не закончится.

- Каждый вызов занимает кадр в стеке, поэтому глубина ограничена:
  по умолчанию 1000 (`sys.getrecursionlimit()`). Ядро Jupyter поднимает
  предел до 3000 – это число и появится в выводе ниже.
- При превышении – `RecursionError`, а не «зависание».

Рекурсия уместна для **рекурсивных по природе** данных: дерево каталогов,
вложенный JSON, иерархия классов.

In [23]:
import sys


def factorial(n):
    if n <= 1:                 # базовый случай: без него рекурсия не закончится
        return 1
    return n * factorial(n - 1)


print("10! =", factorial(10))
print("предел глубины:", sys.getrecursionlimit())

try:
    factorial(5000)
except RecursionError as e:
    print("RecursionError:", e)


10! = 3628800
предел глубины: 3000
RecursionError: maximum recursion depth exceeded


### Хвостовая рекурсия

Как разворачивается вычисление `4!` в двух версиях:

```text
factorial(4)                         factorial_tail(4)
= 4 * factorial(3)                   = factorial_tail(3, acc=4)
= 4 * (3 * factorial(2))             = factorial_tail(2, acc=12)
= 4 * (3 * (2 * factorial(1)))       = factorial_tail(1, acc=24)
= 4 * (3 * (2 * 1))                  = 24
= 24
```

В `factorial` после возврата из вложенного вызова остаётся работа – умножить
на `n`. Каждый вызов ждёт следующего, и цепочка растёт.

В `factorial_tail` рекурсивный вызов – **последнее действие** функции, его
результат сразу возвращается. Такой вызов называют **хвостовым**, а промежуточный
результат передают в аргументе-накопителе `acc`. Вызову нечего ждать, поэтому
Scheme, Scala и Kotlin (с пометкой `tailrec`) превращают такую рекурсию в цикл.

Python этого сознательно не делает: при ошибке в трассировке должны быть видны
все вызовы. Хвостовая версия падает на той же глубине, поэтому там, где
естественно написать цикл, пишут цикл: он быстрее и не ограничен глубиной.

In [30]:
# Хвостовая версия: умножение перенесено в накопитель acc
def factorial_tail(n, acc=1):
    if n <= 1:
        return acc
    return factorial_tail(n - 1, acc * n)    # хвостовой вызов: после него ничего не делаем


print("4! =", factorial_tail(4))
print("совпадает с factorial:", factorial_tail(10) == factorial(10))

try:
    factorial_tail(5000)
except RecursionError as e:
    print("RecursionError:", e)

4! = 24
совпадает с factorial: True
RecursionError: maximum recursion depth exceeded


In [24]:
# Тот же результат циклом: быстрее и без ограничения по глубине
def factorial_loop(n):
    acc = 1
    for k in range(2, n + 1):
        acc *= k
    return acc


print("совпадают:", factorial(20) == factorial_loop(20))


совпадают: True


### Рекурсивные данные: дерево

Вложенный конфиг устроен как дерево. Словари и списки – **узлы**: от них
отходят ветви. Строка, число, `True` – **листья**: внутри них ничего нет,
ветвь на них заканчивается.

```text
config
├── db
│   ├── host: "localhost"     лист
│   └── port: 5432            лист
├── cache
│   └── redis
│       └── ttl: 60           лист
├── admins
│   ├── "anna"                лист
│   └── "boris"               лист
└── debug: True               лист
```

Подсчёт листьев устроен как `factorial`: лист – базовый случай (ответ 1),
у узла ответ – сумма ответов по его ветвям. Глубину вложенности заранее знать
не нужно, а циклом такой обход без собственного стека не написать.

In [32]:
# Считаем листья конфига рекурсивно
config = {"db": {"host": "localhost", "port": 5432},
          "cache": {"redis": {"ttl": 60}},
          "admins": ["anna", "boris"],
          "debug": True}


def count_leaves(node):
    if isinstance(node, dict):          # узел-словарь: сумма по значениям
        return sum(count_leaves(value) for value in node.values())
    if isinstance(node, list):          # узел-список: сумма по элементам
        return sum(count_leaves(item) for item in node)
    return 1                            # базовый случай: лист


print("листьев в конфиге:", count_leaves(config))

листьев в конфиге: 6


### Аннотации и докстринги

Две вещи, которые превращают функцию из «работает» в «с ней можно работать».

- **Аннотации** указывают ожидаемые типы параметров и результата:
  `def f(x: int) -> str`.
- **Докстринг** – строка сразу под заголовком функции. Она попадает в `__doc__`
  и в `help()`: описание собственной функции доступно так же, как описание
  функций стандартной библиотеки.

Обе доступны во время выполнения: `f.__annotations__` и `f.__doc__`.

Контракт «индекс или его отсутствие» из блока про `return` записывается
как `-> int | None`: аннотация делает договорённость видимой в сигнатуре,
не заставляя читать докстринг.

In [25]:
def discount(price: float, percent: int = 10) -> float:
    """Вернуть цену со скидкой.

    price   – исходная цена, percent – размер скидки в процентах.
    Скидка вне диапазона 0..100 считается ошибкой.
    """
    if not 0 <= percent <= 100:
        raise ValueError("percent должен быть в диапазоне 0..100")
    return round(price * (1 - percent / 100), 2)


print(discount(1000), discount(1000, 25))
print("аннотации:", discount.__annotations__)
print("докстринг:", discount.__doc__.splitlines()[0])

900.0 750.0
аннотации: {'price': <class 'float'>, 'percent': <class 'int'>, 'return': <class 'float'>}
докстринг: Вернуть цену со скидкой.


## Чистые функции и побочные эффекты

Функция **чистая**, если выполнены два условия:

1. **Результат зависит только от аргументов.** Те же аргументы – тот же ответ
   при любом вызове. Глобальные переменные, файлы, текущее время, случайные
   числа функция не читает.
2. **Нет побочных эффектов.** Снаружи ничего не меняется: функция не печатает,
   не пишет в файл, не трогает глобальные переменные и переданные ей объекты.
   Единственный результат её работы – возвращённое значение.

Если нарушено хотя бы одно условие, функция **нечистая** (её ещё называют
«грязной»). Пара `with_tax` и `add_tax_inplace` из блока про передачу
аргументов – чистая и нечистая версии одного расчёта.

Чистую функцию проверяет один `assert`. Нечистую – только вместе с окружением:
подготовить глобальные данные, перехватить вывод. Побочные эффекты при этом
нужны, без них программа ничего не делает. Поэтому их отделяют от вычислений:
считает чистая функция, печатает отдельная.

In [34]:
# Нарушено условие 1: результат зависит не только от аргументов
tax_rate = 1.2


def price_with_tax(price):
    return round(price * tax_rate, 2)    # tax_rate берётся из глобальной переменной


print(price_with_tax(100))
tax_rate = 1.1                           # переменную поменяли в другом месте программы
print(price_with_tax(100))               # тот же аргумент – другой ответ


# Чистая версия: всё, от чего зависит ответ, приходит аргументами
def price_with_tax_pure(price, rate):
    return round(price * rate, 2)


assert price_with_tax_pure(100, 1.2) == 120.0
assert price_with_tax_pure(100, 1.1) == 110.0

120.0
110.0


In [35]:
# Нарушено условие 2: побочные эффекты
log = []

def add_dirty(value):
    log.append(value)                  # меняет внешний список
    print("накоплено:", sum(log))      # печатает; сумма зависит от прошлых вызовов


# Чистая версия: тот же расчёт, но результат возвращается
def add_clean(current, value):
    return current + value


add_dirty(5)
add_dirty(7)

print(add_clean(0, 5), add_clean(5, 7))
assert add_clean(5, 7) == 12      # чистую функцию проверяем одной строкой

накоплено: 5
накоплено: 12
5 12


In [27]:
# Вычисление отдельно, вывод отдельно
def average(numbers):
    return sum(numbers) / len(numbers)


def show_average(numbers):
    print(f"среднее по {len(numbers)} значениям: {average(numbers):.2f}")


assert average([2, 4, 6]) == 4    # проверяем вычисление
show_average([2, 4, 6])           # печать вынесена в отдельную функцию

среднее по 3 значениям: 4.00


## Тестирование функций: основы pytest

Функцию мало написать – надо убедиться, что она работает правильно и **продолжает** работать после изменений. Автоматические тесты снимают эту заботу: ожидаемое поведение описывается один раз, а проверяется одной командой.

Строительный блок уже знаком: инструкция **`assert`** из лекции про инструкции. Дальше нужен только способ собирать такие проверки и запускать их пачкой.

In [28]:
def add(a, b):
    return a + b

# assert: молчит, если условие истинно; падает, если ложно
assert add(2, 3) == 5
assert add(-1, 1) == 0
print("Все проверки пройдены")

# При провале видно, что именно пошло не так
try:
    assert add(2, 2) == 5, "2 + 2 должно быть 4, а не 5"
except AssertionError as e:
    print("Проверка провалена:", e)

Все проверки пройдены
Проверка провалена: 2 + 2 должно быть 4, а не 5


### pytest: тест – это функция `test_*`

**pytest** – самый популярный фреймворк тестирования в Python. Соглашение простое:
- тесты лежат в файлах `test_*.py`;
- каждый тест – функция с именем `test_*`, внутри – обычные `assert`;
- запуск одной командой `pytest`, которая сама находит и выполняет все тесты.

Ниже – мини-раннер, который имитирует то, что pytest делает автоматически.
Тесты он находит через `globals()` – словарь всех имён, созданных на верхнем
уровне ноутбука или модуля: ключ – имя, значение – сам объект. Подробно
о пространствах имён – в следующей лекции.

In [29]:
def test_add_positive():
    assert add(2, 3) == 5

def test_add_with_zero():
    assert add(5, 0) == 5

def test_add_negative():
    assert add(-2, -3) == -5

# Так работает pytest под капотом: найти все test_* и выполнить.
# list() – снимок словаря: цикл сам создаёт глобальные имена name и func,
# а словарь, изменившийся во время обхода, даёт RuntimeError
for name, func in list(globals().items()):
    if name.startswith("test_"):
        try:
            func()
            print(f"PASSED  {name}")
        except AssertionError as e:
            print(f"FAILED  {name}: {e}")

PASSED  test_add_positive
PASSED  test_add_with_zero
PASSED  test_add_negative


### Полезные возможности pytest

- **`@pytest.mark.parametrize`** – один тест на много наборов входных данных (без копипаста).
- **`pytest.raises`** – проверка, что функция выбрасывает нужное исключение.

`pytest.raises` – контекстный менеджер: код внутри `with` **обязан** выбросить
указанное исключение. Выбросил – исключение перехвачено, наружу не вышло,
выполнение продолжается дальше. Не выбросил или выбросил другое – тест падает.
Аргумент `match` дополнительно сверяет текст сообщения с регулярным выражением.

Проверим этими инструментами `discount` из раздела про аннотации: её докстринг
обещает ошибку, если скидка выходит за пределы 0..100.

In [34]:
import pytest

# discount обещает ValueError, если скидка вне диапазона 0..100:
# и 150, и -5 – недопустимые значения, обе проверки ждут исключение.
# Контекстный менеджер перехватывает исключение, наружу оно не выходит –
# следующая строка выполнится
with pytest.raises(ValueError):
    discount(1000, 150)
print("150: ValueError получен, как и ожидалось")

# match – регулярное выражение, которое ищется в тексте исключения
with pytest.raises(ValueError, match="диапазоне"):
    discount(1000, -5)
print("-5: ValueError получен, в тексте есть «диапазоне»")

150: ValueError получен, как и ожидалось
-5: ValueError получен, в тексте есть «диапазоне»


In [35]:
# Обвязка: запускаем pytest отдельным процессом во временной папке
import subprocess
import sys
import tempfile
from pathlib import Path


def run_pytest(files, *options):
    """Запустить pytest во временной папке и вернуть отчёт.

    files – словарь «имя файла → текст файла», options – флаги командной строки pytest.
    """
    with tempfile.TemporaryDirectory() as tmp:
        for name, text in files.items():
            (Path(tmp) / name).write_text(text, encoding="utf-8")
        run = subprocess.run([sys.executable, "-m", "pytest", *options],
                             cwd=tmp, capture_output=True, text=True)
    return run.stdout

In [36]:
# Настоящий запуск pytest: код и тесты лежат в разных файлах
import inspect

TESTS = """
import pytest
from pricing import discount

# Наборы выбраны по границам: 0 и 100 допустимы, 101 – уже ошибка
@pytest.mark.parametrize("percent, expected", [(0, 1000), (25, 750), (100, 0)])
def test_discount(percent, expected):
    assert discount(1000, percent) == expected

def test_discount_over_100():
    with pytest.raises(ValueError):
        discount(1000, 101)
"""

# -v – отчёт по строке на каждый тест, --no-header – без шапки с версиями Python и pytest
# inspect.getsource возвращает исходный текст функции: кладём discount в pricing.py
report = run_pytest({"pricing.py": inspect.getsource(discount), "test_pricing.py": TESTS},
                    "-v", "--no-header")

# Один parametrize дал три отдельных теста – видно в списке
print("\n".join(report.splitlines()[1:9]))

collecting ... collected 4 items

test_pricing.py::test_discount[0-1000] PASSED                            [ 25%]
test_pricing.py::test_discount[25-750] PASSED                            [ 50%]
test_pricing.py::test_discount[100-0] PASSED                             [ 75%]
test_pricing.py::test_discount_over_100 PASSED                           [100%]

============================== 4 passed in 0.00s ===============================


In [36]:
# Красный прогон: как pytest показывает причину падения
# В этой версии pricing.py ошибка: процент не делится на 100
BUGGY = """
def discount(price, percent=10):
    return round(price * (1 - percent), 2)
"""

FAILING = """
from pricing import discount

def test_discount_ten_percent():
    assert discount(1000, 10) == 900
"""

# pytest сам показывает, что сравнивалось и откуда взялось значение.
# BUGGY станет файлом pricing.py, FAILING – файлом test_pricing.py;
# -q – краткий отчёт: по символу на тест, подробности только о падениях
print(run_pytest({"pricing.py": BUGGY, "test_pricing.py": FAILING}, "-q", "--no-header"))

F                                                                        [100%]
=================================== FAILURES ===================================
__________________________ test_discount_ten_percent ___________________________

    def test_discount_ten_percent():
>       assert discount(1000, 10) == 900
E       assert -9000 == 900
E        +  where -9000 = discount(1000, 10)

test_pricing.py:5: AssertionError
=========================== short test summary info ============================
FAILED test_pricing.py::test_discount_ten_percent - assert -9000 == 900
1 failed in 0.04s



### Как запускать pytest

```bash
pip install pytest            # установка (в активированный .venv)

pytest                        # найти и запустить все test_*.py
pytest test_pricing.py        # только один файл
pytest -v                     # подробный вывод по каждому тесту
pytest -q                     # краткий вывод, как в красном прогоне
pytest -k "discount"          # только тесты, в имени которых есть "discount"
```

## Частые ошибки

**Изменяемое значение по умолчанию** – самая известная ловушка Python. Значение
по умолчанию вычисляется **один раз** при определении функции, а не при каждом вызове.
Поэтому один и тот же список «живёт» между вызовами.

**Функция и её вызов – разные вещи.** `func` – сам объект функции, `func()` –
её результат. Лишние скобки там, где ждут функцию, и пропущенные там, где ждут
значение, дают либо `TypeError`, либо молча неверный результат.

In [32]:
# ОПАСНО: изменяемый дефолт создаётся ОДИН раз
def add_item(item, basket=[]):
    basket.append(item)
    return basket

print(add_item("яблоко"))   # ['яблоко']
print(add_item("банан"))    # ['яблоко', 'банан'] – тот же список между вызовами!

# ПРАВИЛЬНО: дефолт None, а список создаём внутри
def add_item_ok(item, basket=None):
    if basket is None:
        basket = []
    basket.append(item)
    return basket

print(add_item_ok("яблоко"))  # ['яблоко']
print(add_item_ok("банан"))   # ['банан'] – каждый вызов с чистого листа

['яблоко']
['яблоко', 'банан']
['яблоко']
['банан']


In [40]:
# Функцию ждут – передаём объект, без скобок
print(sorted(["дом", "а", "кот"], key=len))

# Со скобками len вызывается сразу и падает: аргумента нет
try:
    sorted(["дом", "а", "кот"], key=len())
except TypeError as error:
    print("TypeError:", error)


def total():
    return 42


# Обратная ошибка: забыть скобки там, где нужен результат
print("сам объект вызываемый:", callable(total))
print("результат вызываемый:", callable(total()))
print("нужен результат:", total())

['а', 'дом', 'кот']
TypeError: len() takes exactly one argument (0 given)
сам объект вызываемый: True
результат вызываемый: False
нужен результат: 42


## Мини-упражнения


### 1. `is_palindrome(s)`

`True`, если строка читается одинаково в обе стороны. Учитываются только буквы
и цифры, регистр не важен.
`is_palindrome("А роза упала на лапу Азора") → True`.

In [37]:
def is_palindrome(s):
    letters = "".join(ch for ch in s.lower() if ch.isalnum())
    return letters == letters[::-1]

print(is_palindrome("А роза упала на лапу Азора"))
print(is_palindrome("не палиндром"))

True
False


### 2. `top_three(scores)` без побочного эффекта

Функция возвращает три лучших результата, но портит список вызывающего кода:

```python
def top_three(scores):
    scores.sort(reverse=True)
    return scores[:3]

results = [72, 95, 60, 88, 91]
print(top_three(results))   # [95, 91, 88]
print(results)              # [95, 91, 88, 72, 60] – исходный порядок потерян
```

Исправьте функцию так, чтобы аргумент не менялся. Проверьте `assert`-ами
и результат, и сохранность `results`.

In [40]:
def top_three(scores):
    return sorted(scores, reverse=True)[:3]   # sorted строит новый список


results = [72, 95, 60, 88, 91]
assert top_three(results) == [95, 91, 88]
assert results == [72, 95, 60, 88, 91]        # список вызывающего кода не тронут
assert top_three([40]) == [40]                # меньше трёх результатов – сколько есть
print("проверки пройдены")

проверки пройдены


### 3. `apply_twice(func, x)`

Примените функцию дважды: `func(func(x))`. `apply_twice(lambda n: n + 3, 10) → 16`.
Проверьте себя парой `assert`-ов (как в разделе про `pytest`).

In [42]:
def apply_twice(func, x):
    return func(func(x))


print(apply_twice(lambda n: n + 3, 10))
print(apply_twice(str.upper, "hack"))     # метод str.upper передаётся как функция

assert apply_twice(lambda n: n * 2, 3) == 12
assert apply_twice(str.upper, "hack") == "HACK"
print("проверки пройдены")

16
HACK
проверки пройдены


### 4. `call_signature(name, *args, **kwargs)`

Соберите строку вызова функции по её имени и аргументам:
`call_signature("f", 1, 2, sep="-")` → `f(1, 2, sep='-')`.

Позиционные аргументы идут как есть, именованные – в виде `имя=значение`.
Значения показывайте через `repr` (в f-строке – `{value!r}`), чтобы строки
были в кавычках. Именно так
выглядит вывод логирующих декораторов и отладчиков.


In [36]:
def call_signature(name, *args, **kwargs):
    parts = [repr(a) for a in args]                                   # позиционные
    parts += [f"{key}={value!r}" for key, value in kwargs.items()]    # именованные
    return f"{name}({', '.join(parts)})"


print(call_signature("f", 1, 2, sep="-"))
print(call_signature("open", "data.csv", mode="r", encoding="utf-8"))
print(call_signature("now"))          # без аргументов – пустые скобки

assert call_signature("g", [1, 2]) == "g([1, 2])"
print("проверки пройдены")

f(1, 2, sep='-')
open('data.csv', mode='r', encoding='utf-8')
now()
проверки пройдены


### 5. Последний крупный заказ

Напишите `find_last_big_order(amounts, min_sum)`:

- вернуть индекс **последнего** заказа на сумму не меньше `min_sum`;
- вернуть `None`, если таких заказов нет, в том числе на пустом входе;
- выбросить `ValueError`, если `min_sum < 1`.

Пример: `find_last_big_order([180, 940, 210, 760], 500)` → `3`.
Проверьте также единственное совпадение с индексом `0`.

In [8]:
def find_last_big_order(amounts, min_sum):
    if min_sum < 1:
        raise ValueError("min_sum должен быть не меньше 1")
    last_index = None
    for index, amount in enumerate(amounts):
        if amount >= min_sum:
            last_index = index
    return last_index


assert find_last_big_order([180, 940, 210, 760], 500) == 3
assert find_last_big_order([940, 210], 500) == 0
assert find_last_big_order([210], 500) is None
assert find_last_big_order([], 500) is None
with pytest.raises(ValueError):
    find_last_big_order([210], 0)
print("Все пять проверок пройдены")

Все пять проверок пройдены


## Что почитать

- [Определение функций](https://docs.python.org/3/tutorial/controlflow.html) – аргументы,
  значения по умолчанию, `*args`/`**kwargs`.
- [PEP 257](https://peps.python.org/pep-0257/) – как оформлять докстринги.
- [Документация pytest](https://docs.pytest.org/en/stable/) – фикстуры, параметризация,
  запуск отдельных тестов.
- Л. Рамальо, «Python. К вершинам мастерства» – глава о функциях как объектах
  первого класса.